# PMIP — Data Cleaning

## Purpose

This notebook prepares the raw PMIP datasets for analysis, integration,
feature engineering, and machine learning.

The cleaning process will be carried out carefully so that important
information is not removed or modified without first understanding why.

For each dataset, the cleaning process will include:

- inspecting the dataset structure;
- identifying missing values;
- checking for duplicate records;
- checking and correcting data types;
- standardising text values where necessary;
- identifying invalid or unusual values;
- validating the cleaned dataset; and
- saving the processed data for later stages of the PMIP project.

The first dataset to be cleaned is `listeners.csv`.

## Table of Contents

1. [Listeners Dataset Cleaning](#1-listeners-dataset)
   - [1.1 Initial Dataset Inspection](#11-initial-dataset-inspection)
   - [1.2 Data Type Inspection](#12-data-type-inspection)
   - [1.3 Numeric Column Cleaning](#13-cleaning-numeric-listener-columns)
   - [1.4 Numeric Value Validation](#14-numeric-value-validation)
   - [1.5 Artist Name Validation](#15-artist-name-validation)
   - [1.6 Final Validation](#16-final-validation-of-the-cleaned-listeners-dataset)
   - [1.7 Save and Verify](#17-saving-and-verifying-the-cleaned-listeners-dataset)
   - [1.8 Cleaning Summary](#18-listeners-dataset-cleaning-summary)
2. [Cross-Platform Track Dataset Cleaning](#2-cross-platform-track-dataset)
   - [2.1 Initial Dataset Inspection](#21-initial-dataset-inspection)
   - [2.2 Duplicate and ISRC Validation](#22-duplicate-and-isrc-validation)
   - [2.3 Missing Artist Values](#23-investigating-missing-artist-values)
   - [2.4 Missing Platform Metrics](#24-investigating-missing-platform-metrics)
   - [2.5 Numeric Data Type Cleaning](#25-investigating-numeric-data-types)
   - [2.6 Release Date Cleaning](#26-cleaning-the-release-date-column)
   - [2.7 All Time Rank Cleaning](#27-cleaning-the-all-time-rank-column)
   - [2.8 Text and Whitespace Cleaning](#28-checking-text-columns-for-whitespace-issues)
   - [2.9 Encoding Investigation](#29-investigating-text-encoding-issues)
   - [2.10 Numeric Range and Missingness Analysis](#210-numeric-range-and-missingness-analysis)
   - [2.11 Final Validation and Save](#211-final-validation-and-save)
3. [Charts Dataset Cleaning](#3-charts-dataset-cleaning)
   - [3.1 Loading and Initial Inspection](#31-loading-and-initial-inspection)
   - [3.2 Dataset Structure and Data Types](#32-dataset-structure-and-data-types)
   - [3.3 Missing Values](#33-missing-values-inspection)
   - [3.4 Duplicate Rows](#34-duplicate-row-inspection)
   - [3.5 Date Validation](#35-date-validation-and-conversion)
   - [3.6 Country, Position and Stream Validation](#36-country-chart-position-and-stream-validation)
   - [3.7 Track Identifier and Text Validation](#37-track-identifier-and-text-field-validation)
   - [3.8 Duration and Explicit Content Validation](#38-duration-and-explicit-content-validation)
   - [3.9 Categorical and Country Code Validation](#39-categorical-and-country-code-validation)
   - [3.10 Final Validation and Save](#310-final-charts-dataset-validation)
4. [Data Cleaning Summary](#4-data-cleaning-summary)


## 1. Listeners Dataset

The `listeners.csv` dataset contains artist-level listener and popularity
information that will later contribute to PMIP's artist intelligence features.

Before modifying the dataset, an initial data-quality assessment will be
performed to understand its structure and identify potential cleaning issues.

In [1]:
import pandas as pd

In [2]:
listeners_df = pd.read_csv("../data/raw/listeners.csv")

listeners_df.head()

,Artist,Listeners,Daily Trend,Peak,PkListeners
0,The Weeknd,"107,592,328","-138,880",1,"113,034,886"
1,Taylor Swift,"101,003,302",889,2,"101,003,302"
2,Ed Sheeran,"76,475,126","-68,137",2,"87,934,910"
3,Dua Lipa,"76,421,916","-71,356",4,"77,778,397"
4,Bad Bunny,"76,162,057","-199,052",3,"83,950,570"


### 1.1 Initial Dataset Inspection

Before making any changes, the structure of the listeners dataset will be
examined.

This inspection will identify:

- the number of rows and columns;
- the available column names;
- the data type assigned to each column; and
- whether any columns may require type conversion during cleaning.

No data will be modified at this stage.

In [3]:
listeners_df.shape

(2500, 5)

#### Dataset Size

The listeners dataset contains **2,500 rows and 5 columns**.

Each row represents an artist record, while the columns contain information
about the artist's listener statistics and ranking performance.

In [4]:
listeners_df.columns.tolist()

['Artist', 'Listeners', 'Daily Trend', 'Peak', 'PkListeners']

### 1.2 Data Type Inspection

The data types of each column are inspected before cleaning.

Correct data types are important because numerical calculations, statistical
analysis, visualisation, and machine learning require values to be stored in
an appropriate format.

For example, listener counts may appear numeric when displayed but could
actually be stored as text because the values contain commas.

In [5]:
listeners_df.dtypes

Artist           str
Listeners        str
Daily Trend      str
Peak           int64
PkListeners      str
dtype: object

#### Data Type Findings

The initial inspection identified a data type issue in three numerical columns.

`Listeners`, `Daily Trend`, and `PkListeners` are currently stored as strings
rather than numerical values. Inspection of the dataset suggests that this is
caused by comma separators within values such as `107,592,328`.

The `Peak` column has already been correctly recognised as an integer.

The affected columns will be converted to appropriate numerical data types
during the cleaning stage after further data-quality checks have been completed.

In [6]:
listeners_df.isnull().sum()

Artist         0
Listeners      0
Daily Trend    0
Peak           0
PkListeners    0
dtype: int64

#### Missing Value Findings

No missing values were identified in the listeners dataset.

All five columns contain values for all 2,500 records. Therefore, no rows
need to be removed and no missing values need to be filled during this
stage of the cleaning process.

In [7]:
listeners_df.duplicated().sum()

np.int64(0)

#### Duplicate Row Findings

No completely duplicated rows were identified in the listeners dataset.

All 2,500 records are unique when comparing the values across all five
columns. Therefore, no rows need to be removed at this stage.

In [8]:
listeners_df["Artist"].duplicated().sum()

np.int64(0)

#### Artist Duplicate Findings

No duplicated artist names were found in the dataset.

Each of the 2,500 records represents a unique artist. This is useful for
later dataset integration because each artist has only one listener record.

In [9]:
listeners_df[["Listeners", "Daily Trend", "PkListeners"]].head(10)

,Listeners,Daily Trend,PkListeners
0,"107,592,328","-138,880","113,034,886"
1,"101,003,302",889,"101,003,302"
2,"76,475,126","-68,137","87,934,910"
3,"76,421,916","-71,356","77,778,397"
4,"76,162,057","-199,052","83,950,570"
5,"75,784,389","116,405","80,958,750"
6,"75,371,611","-67,013","76,391,086"
7,"72,623,228","34,540","75,467,229"
8,"71,793,820","-72,877","72,368,528"
9,"71,277,599","71,089","84,140,935"


### 1.3 Cleaning Numeric Listener Columns

The `Listeners`, `Daily Trend`, and `PkListeners` columns were imported as
strings rather than numeric values.

Inspection of the data shows that these columns contain commas used as
thousands separators, for example `107,592,328` and `-138,880`.

The commas will therefore be removed and the columns converted to numeric
data types so that they can be used correctly in statistical analysis,
visualisation, and machine learning.


In [10]:
numeric_columns = ["Listeners", "Daily Trend", "PkListeners"]

for column in numeric_columns:
    listeners_df[column] = (
        listeners_df[column]
        .str.replace(",", "", regex=False)
        .astype("int64")
    )

In [11]:
listeners_df.dtypes

Artist           str
Listeners      int64
Daily Trend    int64
Peak           int64
PkListeners    int64
dtype: object

### 1.4 Numeric Value Validation

After converting the listener metrics to numeric data types, the next step is
to inspect their ranges.

This helps identify potentially unrealistic values or outliers while avoiding
the removal of valid values, such as negative `Daily Trend` values that
represent a decrease in listeners.


In [12]:
listeners_df.describe()

,Listeners,Daily Trend,Peak,PkListeners
count,2.500000e+03,2500.00000,2500.0000,2.500000e+03
mean,1.111836e+07,4801.96920,1061.2228,1.208679e+07
std,1.022231e+07,58063.47358,639.7277,1.111030e+07
min,4.274633e+06,-440555.00000,1.0000,4.285059e+06
25%,5.400151e+06,-13662.50000,516.0000,5.900046e+06
50%,7.463490e+06,67.50000,1034.5000,8.169010e+06
75%,1.203113e+07,15556.00000,1582.2500,1.317932e+07
max,1.075923e+08,743072.00000,2493.0000,1.130349e+08


### Numeric Validation Findings

The numeric summary shows that all four numeric columns contain 2,500 valid
values, confirming that no numeric values are missing.

The listener-related columns contain positive values within plausible ranges.
The `Daily Trend` column contains both positive and negative values. These
values are retained because negative values represent decreases in listener
counts rather than invalid data.

Some large positive and negative `Daily Trend` values may represent outliers.
However, they will not be removed at this stage because unusual changes in
listener activity may contain useful information about artist momentum and
popularity.

The `Peak` column ranges from 1 to 2,493, which is consistent with its use as
a ranking variable.

No values were removed during this validation step.

### 1.5 Artist Name Validation

The `Artist` column will be inspected for formatting inconsistencies before
standardisation.

Artist names are important because they will later be used to identify and
match artists across the PMIP datasets. Differences in capitalisation,
leading or trailing spaces, or other formatting inconsistencies could prevent
the same artist from being matched correctly.


In [13]:
listeners_df["Artist"].head(20)

0        The Weeknd
1      Taylor Swift
2        Ed Sheeran
3          Dua Lipa
4         Bad Bunny
5           Rihanna
6             Drake
7     Justin Bieber
8     Billie Eilish
9       Miley Cyrus
10     David Guetta
11     Travis Scott
12    Ariana Grande
13         Coldplay
14    Calvin Harris
15          Shakira
16           Eminem
17      Post Malone
18       Bruno Mars
19         Doja Cat
Name: Artist, dtype: str

### Checking Artist Names for Extra Whitespace

The first 20 artist names appear correctly formatted. However, visual inspection
alone cannot detect all formatting problems across the dataset.

The full `Artist` column will therefore be checked for leading or trailing
whitespace. These hidden spaces could cause matching problems when artist names
are compared across the PMIP datasets.

In [14]:
whitespace_count = (
    listeners_df["Artist"] != listeners_df["Artist"].str.strip()
).sum()

whitespace_count

np.int64(0)

### Checking Artist Names for Case Inconsistencies

Artist names may appear unique while still representing the same artist if
different capitalisation is used.

The artist names will therefore be compared in lowercase form to determine
whether any duplicate identities appear when capitalisation is ignored.

In [15]:
case_duplicate_count = (
    listeners_df["Artist"]
    .str.lower()
    .duplicated()
    .sum()
)

case_duplicate_count

np.int64(0)

### Artist Name Validation Findings

The `Artist` column contains no missing values, duplicated artist names,
leading or trailing whitespace, or duplicate identities caused by differences
in capitalisation.

The original artist names will therefore be preserved without modification.

For later dataset integration, a separate normalised artist-name field may be
created specifically for matching purposes rather than altering the original
display names.

### 1.6 Final Validation of the Cleaned Listeners Dataset

Before saving the cleaned dataset, a final validation is performed to confirm
that the cleaning process has not introduced any new problems.

The final checks include:

- confirming the dataset dimensions;
- checking for missing values;
- checking for duplicate rows; and
- confirming the final data types.

This ensures that the processed dataset is ready for later analysis,
integration, feature engineering, and machine learning.


In [16]:
print("Shape:")
print(listeners_df.shape)

print("\nMissing values:")
print(listeners_df.isnull().sum())

print("\nDuplicate rows:")
print(listeners_df.duplicated().sum())

print("\nData types:")
print(listeners_df.dtypes)

Shape:
(2500, 5)

Missing values:
Artist         0
Listeners      0
Daily Trend    0
Peak           0
PkListeners    0
dtype: int64

Duplicate rows:
0

Data types:
Artist           str
Listeners      int64
Daily Trend    int64
Peak           int64
PkListeners    int64
dtype: object


### 1.7 Saving and Verifying the Cleaned Listeners Dataset

The cleaned listeners dataset has passed the final validation checks.

A processed copy will now be saved separately from the original raw data.
Keeping raw and processed data separate preserves the original dataset while
providing a clean version for later data integration, exploratory analysis,
feature engineering, visualisation, and machine learning.


In [17]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

listeners_df.to_csv(
    processed_dir / "listeners_cleaned.csv",
    index=False
)

print("Cleaned listeners dataset saved successfully.")

Cleaned listeners dataset saved successfully.


## Verifying the Saved Processed Dataset

The processed listeners dataset is loaded back into Pandas to confirm that it
was saved correctly and can be read without errors.

The verification checks the dataset shape, data types, missing values, and
duplicate records after export.

In [18]:
listeners_cleaned_df = pd.read_csv(
    "../data/processed/listeners_cleaned.csv"
)

print("Shape:")
print(listeners_cleaned_df.shape)

print("\nMissing values:")
print(listeners_cleaned_df.isnull().sum())

print("\nDuplicate rows:")
print(listeners_cleaned_df.duplicated().sum())

print("\nData types:")
print(listeners_cleaned_df.dtypes)

Shape:
(2500, 5)

Missing values:
Artist         0
Listeners      0
Daily Trend    0
Peak           0
PkListeners    0
dtype: int64

Duplicate rows:
0

Data types:
Artist           str
Listeners      int64
Daily Trend    int64
Peak           int64
PkListeners    int64
dtype: object


### 1.8 Listeners Dataset Cleaning Summary

The `listeners.csv` dataset has been successfully cleaned and validated.

The cleaning process confirmed that:

- the dataset contains 2,500 unique artist records;
- no missing values are present;
- no duplicate rows or duplicate artist names are present;
- artist names contain no leading or trailing whitespace;
- no duplicate artist identities were found when capitalisation was ignored;
- `Listeners`, `Daily Trend`, and `PkListeners` were converted from strings to integer values;
- valid negative `Daily Trend` values were retained;
- no numeric values were removed during validation; and
- the cleaned dataset was successfully saved and reloaded from
  `data/processed/listeners_cleaned.csv`.

The cleaned listeners dataset is now ready for later integration, exploratory
analysis, feature engineering, and machine learning.


# 2. Cross-Platform Track Dataset

The next dataset contains track-level performance information across multiple
music and social platforms, including Spotify, YouTube, TikTok, Apple Music,
Deezer, Pandora, SoundCloud, Shazam, and radio-related metrics.

Before cleaning, the dataset will be loaded and profiled to identify:

- its dimensions;
- missing values;
- duplicate rows;
- incorrect data types;
- completely empty columns;
- date formatting issues; and
- numeric values that are currently stored as text.

In [19]:
spotify_df = pd.read_csv(
    "../data/raw/Most Streamed Spotify Songs 2024.csv.zip",
    compression="zip",
    encoding="latin-1"
)

spotify_df.head()

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1


### 2.1 Initial Dataset Inspection

Before making any changes, the dataset will be inspected to understand its
structure and current condition.

The first check examines the number of rows and columns in the dataset.
This establishes the size of the raw dataset before any cleaning operations
are performed.


In [20]:
spotify_df.shape

(4600, 29)

### Inspecting Column Names

The dataset contains 29 columns. The column names will now be inspected to
understand the available variables and identify any naming or formatting
issues that may need to be addressed during cleaning.

In [21]:
spotify_df.columns.tolist()

['Track',
 'Album Name',
 'Artist',
 'Release Date',
 'ISRC',
 'All Time Rank',
 'Track Score',
 'Spotify Streams',
 'Spotify Playlist Count',
 'Spotify Playlist Reach',
 'Spotify Popularity',
 'YouTube Views',
 'YouTube Likes',
 'TikTok Posts',
 'TikTok Likes',
 'TikTok Views',
 'YouTube Playlist Reach',
 'Apple Music Playlist Count',
 'AirPlay Spins',
 'SiriusXM Spins',
 'Deezer Playlist Count',
 'Deezer Playlist Reach',
 'Amazon Playlist Count',
 'Pandora Streams',
 'Pandora Track Stations',
 'Soundcloud Streams',
 'Shazam Counts',
 'TIDAL Popularity',
 'Explicit Track']

### Inspecting Data Types

The data types of all 29 columns will now be examined.

This is particularly important for the platform performance metrics because
values such as streams, views, likes, playlist counts, and playlist reach
should normally be numeric. If these values have been loaded as text, they
will need to be converted before statistical analysis, visualisation, or
machine learning can be performed.

The release date will also be inspected to determine whether it needs to be
converted into a proper date format.

In [22]:
spotify_df.dtypes

Track                             str
Album Name                        str
Artist                            str
Release Date                      str
ISRC                              str
All Time Rank                     str
Track Score                   float64
Spotify Streams                   str
Spotify Playlist Count            str
Spotify Playlist Reach            str
Spotify Popularity            float64
YouTube Views                     str
YouTube Likes                     str
TikTok Posts                      str
TikTok Likes                      str
TikTok Views                      str
YouTube Playlist Reach            str
Apple Music Playlist Count    float64
AirPlay Spins                     str
SiriusXM Spins                    str
Deezer Playlist Count         float64
Deezer Playlist Reach             str
Amazon Playlist Count         float64
Pandora Streams                   str
Pandora Track Stations            str
Soundcloud Streams                str
Shazam Count

### Checking Missing Values

The dataset will now be checked for missing values across all 29 columns.

This is especially important for the cross-platform metrics because not every
track may have data available from every music or social media platform.

The number of missing values in each column will be examined before deciding
how they should be handled. Missing values will not automatically be removed,
as the absence of data may reflect differences in platform coverage rather
than poor data quality.

In [23]:
spotify_df.isnull().sum().sort_values(ascending=False)

TIDAL Popularity              4600
Soundcloud Streams            3333
SiriusXM Spins                2123
Pandora Track Stations        1268
TikTok Posts                  1173
Pandora Streams               1106
Amazon Playlist Count         1055
YouTube Playlist Reach        1009
TikTok Views                   981
TikTok Likes                   980
Deezer Playlist Reach          928
Deezer Playlist Count          921
Spotify Popularity             804
Shazam Counts                  577
Apple Music Playlist Count     561
AirPlay Spins                  498
YouTube Likes                  315
YouTube Views                  308
Spotify Streams                113
Spotify Playlist Reach          72
Spotify Playlist Count          70
Artist                           5
Track                            0
Album Name                       0
Track Score                      0
All Time Rank                    0
ISRC                             0
Release Date                     0
Explicit Track      

### 2.2 Duplicate and ISRC Validation


### Checking for Duplicate Rows

Duplicate records can distort analysis by causing the same track information
to be counted more than once.

The dataset will therefore be checked for rows that are completely identical.
This initial check only identifies exact duplicate rows. Track-level
duplication based on identifiers such as ISRC will be investigated separately.

In [24]:
spotify_df.duplicated().sum()

np.int64(2)

### Inspecting Duplicate Rows

The duplicate check identified 2 rows that are exact duplicates of other
records in the dataset.

Before removing them, the duplicated records will be inspected to confirm
that they represent repeated observations rather than legitimate separate
records.

In [25]:
spotify_df[spotify_df.duplicated(keep=False)].sort_values(
    by=["Track", "Artist"]
)

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
3447,Dembow,Dembow,Danny Ocean,12/8/2017,USWL11700269,"3,441",23.3,"579,189,526","60,397","11,805,084",...,9,20.0,"37,649",12.0,"6,723,858","7,832",NaN,"1,619,550",NaN,0
3450,Dembow,Dembow,Danny Ocean,12/8/2017,USWL11700269,"3,441",23.3,"579,189,526","60,397","11,805,084",...,9,20.0,"37,649",12.0,"6,723,858","7,832",NaN,"1,619,550",NaN,0
2449,Tennessee Orange,Tennessee Orange,Megan Moroney,9/2/2022,TCAGJ2289254,"2,424",28.9,"227,893,586","28,139","12,480,714",...,34,5.0,"1,370",49.0,"56,972,562","26,968","1,336,043","708,143",NaN,0
2450,Tennessee Orange,Tennessee Orange,Megan Moroney,9/2/2022,TCAGJ2289254,"2,424",28.9,"227,893,586","28,139","12,480,714",...,34,5.0,"1,370",49.0,"56,972,562","26,968","1,336,043","708,143",NaN,0


### Removing Exact Duplicate Rows

The duplicate inspection confirmed two pairs of completely identical records:
`Dembow` by Danny Ocean and `Tennessee Orange` by Megan Moroney.

Because each duplicated record contains identical information across all
29 columns, retaining both copies would cause the tracks to be counted more
than once during analysis.

The additional copies will therefore be removed while retaining one valid
record from each pair.

In [26]:
spotify_df = spotify_df.drop_duplicates().reset_index(drop=True)

spotify_df.shape

(4598, 29)

In [27]:
spotify_df.duplicated().sum()

np.int64(0)

### Checking for Duplicate ISRCs

Although exact duplicate rows have been removed, duplicate track identifiers
may still exist.

The ISRC (International Standard Recording Code) identifies a specific sound
recording and can therefore provide a stronger method of identifying repeated
recordings than track titles alone.

Repeated ISRC values will first be counted before any decision is made about
whether the corresponding records should be removed or retained.

In [28]:
duplicate_isrc_count = spotify_df["ISRC"].duplicated().sum()

duplicate_isrc_count

np.int64(0)

### ISRC Duplicate Findings

No duplicate ISRC values were identified after removing the exact duplicate
rows.

This indicates that each remaining record represents a unique sound recording
according to its International Standard Recording Code. No additional rows
need to be removed based on ISRC duplication.

In [29]:
spotify_df[spotify_df["Artist"].isnull()][
    ["Track", "Album Name", "Artist", "ISRC", "Release Date"]
]

,Track,Album Name,Artist,ISRC,Release Date
311,Cool,JnD Mix,NaN,QZNWQ2410638,5/25/2024
480,I Wanna Party,I Wanna Party - Single,NaN,QZYFZ2445017,5/31/2024
1345,Marlboro Remix,Marlboro Remix - Single,NaN,QZNWT2471497,6/7/2024
1561,Melting,Melting - Single,NaN,QZNWU2402635,6/10/2024
3401,La ï¿½ï¿½ltima Vez (Yo Te Per,La ï¿½ï¿½ltima Vez (Yo Te Perdï¿½ï¿½),NaN,MX2832415361,5/2/2024


### 2.3 Investigating Missing Artist Values

Five records contain missing values in the `Artist` column.

These records will not be removed immediately because they still contain
potentially useful information, including track names, ISRC identifiers,
release dates, and platform performance metrics.

The records will first be investigated to determine whether the missing artist
information can be recovered reliably.


In [30]:
missing_artist_rows = spotify_df[spotify_df["Artist"].isnull()]

missing_artist_rows.T

,311,480,1345,1561,3401
Track,Cool,I Wanna Party,Marlboro Remix,Melting,La ï¿½ï¿½ltima Vez (Yo Te Per
Album Name,JnD Mix,I Wanna Party - Single,Marlboro Remix - Single,Melting - Single,La ï¿½ï¿½ltima Vez (Yo Te Perdï¿½ï¿½)
Artist,NaN,NaN,NaN,NaN,NaN
Release Date,5/25/2024,5/31/2024,6/7/2024,6/10/2024,5/2/2024
ISRC,QZNWQ2410638,QZYFZ2445017,QZNWT2471497,QZNWU2402635,MX2832415361
All Time Rank,311,482,"1,343","1,553","3,381"
Track Score,86.5,70.3,40.6,37.2,23.6
Spotify Streams,NaN,NaN,NaN,NaN,NaN
Spotify Playlist Count,NaN,NaN,NaN,NaN,NaN
Spotify Playlist Reach,NaN,NaN,NaN,NaN,NaN


### Assessing Completeness of Records with Missing Artists

Inspection shows that the five records with missing artist names also contain
many missing platform-performance measurements.

Before deciding whether to retain, repair, or remove these records, the number
of missing values in each record will be calculated. This provides an
evidence-based way of assessing how complete each record is rather than
removing records solely because the artist name is missing.

In [31]:
missing_artist_analysis = spotify_df[spotify_df["Artist"].isnull()].copy()

missing_artist_analysis["Missing Values"] = (
    missing_artist_analysis.isnull().sum(axis=1)
)

missing_artist_analysis["Available Values"] = (
    spotify_df.shape[1] - missing_artist_analysis["Missing Values"]
)

missing_artist_analysis[
    ["Track", "ISRC", "Missing Values", "Available Values"]
]

,Track,ISRC,Missing Values,Available Values
311,Cool,QZNWQ2410638,21,8
480,I Wanna Party,QZYFZ2445017,22,7
1345,Marlboro Remix,QZNWT2471497,21,8
1561,Melting,QZNWU2402635,21,8
3401,La ï¿½ï¿½ltima Vez (Yo Te Per,MX2832415361,21,8


## Removing Highly Incomplete Records

Five records were identified with missing values in the `Artist` column.

Further investigation showed that these records were substantially incomplete,
with 21–22 of the 29 available fields missing (approximately 72–76% of each
record). Most of the missing fields relate to platform-performance metrics such
as Spotify streams, playlist reach, YouTube engagement, TikTok activity,
Apple Music playlists, and other music-platform measurements.

Although some identifying information such as track name, release date, and
ISRC remained available, the records contain insufficient information for the
cross-platform analysis planned for PMIP.

The five highly incomplete records will therefore be removed rather than
attempting to impute or manually reconstruct a large proportion of their
missing information.

In [32]:
rows_before = spotify_df.shape[0]

spotify_df = spotify_df.dropna(subset=["Artist"]).reset_index(drop=True)

rows_after = spotify_df.shape[0]

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before: 4598
Rows after: 4593
Rows removed: 5


In [33]:
spotify_df["Artist"].isnull().sum()

np.int64(0)

## Reassessing Missing Platform Metrics

After removing exact duplicate records and the five highly incomplete records,
the missing values across the remaining dataset are reassessed.

Platform-level missing values require careful treatment because the absence of
a metric does not necessarily mean that a track is invalid. Different music
platforms may have different levels of coverage within the dataset.

The missing-value distribution will therefore be examined before deciding
whether individual columns should be retained, removed, or treated using
another cleaning strategy.

In [34]:
missing_summary = pd.DataFrame({
    "Missing Values": spotify_df.isnull().sum(),
    "Missing Percentage": (
        spotify_df.isnull().sum() / len(spotify_df) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values("Missing Percentage", ascending=False)

missing_summary

,Missing Values,Missing Percentage
TIDAL Popularity,4593,100.00
Soundcloud Streams,3327,72.44
SiriusXM Spins,2118,46.11
Pandora Track Stations,1263,27.50
TikTok Posts,1168,25.43
Pandora Streams,1101,23.97
Amazon Playlist Count,1050,22.86
YouTube Playlist Reach,1004,21.86
TikTok Views,976,21.25
TikTok Likes,975,21.23


## Removing Columns with No Usable Data

The missing-value analysis identified `TIDAL Popularity` as completely empty,
with 4,593 missing values across all 4,593 remaining records (100% missing).

Because the column contains no observations, it cannot contribute to
statistical analysis, visualisation, feature engineering, or machine learning.

The `TIDAL Popularity` column will therefore be removed from the cleaned
dataset. Other columns containing partial missing data will be retained for
further investigation because they still contain potentially useful
information.

In [35]:
spotify_df = spotify_df.drop(columns=["TIDAL Popularity"])

spotify_df.shape

(4593, 28)

In [36]:
"TIDAL Popularity" in spotify_df.columns

False

### 2.4 Investigating Missing Platform Metrics

The remaining missing values occur primarily within platform-performance
metrics. These missing values will not automatically be replaced with zero,
because a missing observation does not necessarily indicate zero activity.

For example, a missing Spotify stream count may indicate that the value was
not recorded rather than that the track received zero streams.

To better understand how the dataset represents platform activity, the number
of explicit zero values in each platform metric will be examined and compared
with the number of missing values.


In [37]:
platform_columns = [
    "Spotify Streams",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "Spotify Popularity",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "Apple Music Playlist Count",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Pandora Streams",
    "Pandora Track Stations",
    "Soundcloud Streams",
    "Shazam Counts"
]

In [38]:
platform_summary = pd.DataFrame({
    "Missing Values": spotify_df[platform_columns].isnull().sum(),
    "Zero Values": (spotify_df[platform_columns] == 0).sum()
})

platform_summary["Missing Percentage"] = (
    platform_summary["Missing Values"] / len(spotify_df) * 100
).round(2)

platform_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

,Missing Values,Zero Values,Missing Percentage
Soundcloud Streams,3327,0,72.44
SiriusXM Spins,2118,0,46.11
Pandora Track Stations,1263,0,27.50
TikTok Posts,1168,0,25.43
Pandora Streams,1101,0,23.97
Amazon Playlist Count,1050,0,22.86
YouTube Playlist Reach,1004,0,21.86
TikTok Views,976,0,21.25
TikTok Likes,975,0,21.23
Deezer Playlist Reach,923,0,20.10


### Decision on Missing Platform Metrics

The investigation found that none of the platform-performance columns contain
explicit zero values. However, this does not provide sufficient evidence that
missing values represent zero platform activity.

A missing value may instead indicate that the measurement was unavailable,
not collected, or not represented in the source dataset. Replacing these
values with zero could therefore introduce artificial observations and distort
later statistical analysis or machine learning models.

The remaining missing platform metrics will consequently be preserved as
missing values at this stage. Missing-data treatment will be reconsidered
during feature selection and model preparation, where an appropriate strategy
can be chosen based on the variables and analytical task being used.

### 2.5 Investigating Numeric Data Types

Several platform-performance columns were imported as text (`str`) even though
they represent numerical measurements such as streams, views, likes, playlist
counts, and radio spins.

Before converting these columns to numeric data types, sample values will be
examined to identify formatting characters such as commas that may have caused
Pandas to interpret the values as text.


In [39]:
string_numeric_columns = [
    "Spotify Streams",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Reach",
    "Pandora Streams",
    "Pandora Track Stations",
    "Soundcloud Streams",
    "Shazam Counts"
]

spotify_df[string_numeric_columns].head(10)

,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,YouTube Playlist Reach,AirPlay Spins,SiriusXM Spins,Deezer Playlist Reach,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts
0,"390,470,936","30,716","196,631,588","84,274,754","1,713,126","5,767,700","651,565,900","5,332,281,936","150,597,040","40,975",684,"17,598,718","18,004,655","22,931","4,818,457","2,669,262"
1,"323,703,884","28,113","174,597,137","116,347,040","3,486,739","674,700","35,223,547","208,339,025","156,380,351","40,778",3,"10,422,430","7,780,028","28,444","6,623,075","1,118,279"
2,"601,309,283","54,331","211,607,669","122,599,116","2,228,730","3,025,400","275,154,237","3,369,120,610","373,784,955","74,333",536,"36,321,847","5,022,621","5,639","7,208,651","5,285,340"
3,"2,031,280,633","269,802","136,569,078","1,096,100,899","10,629,796","7,189,811","1,078,757,968","14,603,725,994","3,351,188,582","1,474,799","2,182","24,684,248","190,260,277","203,384",NaN,"11,822,942"
4,"107,034,922","7,223","151,469,874","77,373,957","3,670,188","16,400",NaN,NaN,"112,763,851","12,185",1,"17,660,624","4,493,884","7,006","207,179","457,017"
5,"670,665,438","105,892","175,421,034","131,148,091","1,392,593","4,202,367","214,943,489","2,938,686,633","2,867,222,632","522,042","4,654","17,167,254","138,529,362","50,982","9,438,601","4,517,131"
6,"900,158,751","73,118","201,585,714","308,723,145","4,120,760",NaN,"29,584,940","534,915,313","4,601,579,812","383,478",429,"48,197,850","65,447,476","57,372",NaN,"9,990,302"
7,"675,079,153","40,094","211,236,940","228,382,568","1,439,495","3,500,000","338,546,668","3,804,584,163","2,112,581,620","17,221",30,"33,245,595","3,372,428","5,762",NaN,"6,063,523"
8,"1,653,018,119",1,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,"90,676,573","10,400","184,199,419","32,735,244","988,682","325,800","121,574,500","974,656,200","174,706,874","3,823",117,"10,800,098","1,005,626",842,"3,679,709","666,302"


### Converting Platform Metrics to Numeric Values

Inspection of the platform-performance columns shows that many numerical values
contain commas as thousands separators. For example, Spotify stream counts are
stored in forms such as `390,470,936`.

These formatting characters caused Pandas to interpret several numerical
columns as text rather than numbers.

The commas will therefore be removed and the affected columns converted to
numeric data types. Existing missing values will remain missing rather than
being replaced with zero, preserving the distinction between unavailable data
and genuine platform activity.

In [40]:
for column in string_numeric_columns:
    spotify_df[column] = pd.to_numeric(
        spotify_df[column].str.replace(",", "", regex=False),
        errors="coerce"
    )

In [41]:
spotify_df[string_numeric_columns].dtypes

Spotify Streams           float64
Spotify Playlist Count    float64
Spotify Playlist Reach    float64
YouTube Views             float64
YouTube Likes             float64
TikTok Posts              float64
TikTok Likes              float64
TikTok Views              float64
YouTube Playlist Reach    float64
AirPlay Spins             float64
SiriusXM Spins            float64
Deezer Playlist Reach     float64
Pandora Streams           float64
Pandora Track Stations    float64
Soundcloud Streams        float64
Shazam Counts             float64
dtype: object

### Validating Numeric Conversion

The platform-performance columns have now been converted from text to numeric
data types.

Because invalid values were converted to missing values during this process,
the dataset will now be checked to ensure that the conversion did not
unexpectedly introduce additional missing data.

In [42]:
spotify_df[string_numeric_columns].isnull().sum().sort_values(ascending=False)

Soundcloud Streams        3327
SiriusXM Spins            2118
Pandora Track Stations    1263
TikTok Posts              1168
Pandora Streams           1101
YouTube Playlist Reach    1004
TikTok Views               976
TikTok Likes               975
Deezer Playlist Reach      923
Shazam Counts              576
AirPlay Spins              493
YouTube Likes              310
YouTube Views              303
Spotify Streams            108
Spotify Playlist Reach      67
Spotify Playlist Count      65
dtype: int64

### Numeric Conversion Validation Result

The missing-value counts remained unchanged after converting the platform
performance columns to numeric data types.

This confirms that the conversion was successful and did not introduce any
additional missing values. The comma-formatted numerical values were therefore
converted safely while the original missing values were preserved.

### 2.6 Cleaning the Release Date Column

The `Release Date` column is currently stored as text even though it represents
calendar dates.

Converting this column to a proper datetime format will make it possible to
analyse release timing, extract release years or months, calculate track age,
and study music trends over time.

Before conversion, the existing date values will be inspected to confirm their
format and identify any unexpected values.


In [43]:
spotify_df["Release Date"].head(20)

0      4/26/2024
1       5/4/2024
2      3/19/2024
3      1/12/2023
4      5/31/2024
5     11/10/2023
6      1/18/2024
7       2/2/2024
8       6/9/2024
9      5/23/2024
10     5/10/2024
11     6/14/2024
12     5/17/2024
13     3/22/2024
14     10/7/2022
15     3/22/2023
16     4/18/2024
17     9/14/2023
18     5/17/2024
19     3/31/2022
Name: Release Date, dtype: str

### Validating Release Date Format

The initial sample suggests that release dates use the month/day/year format
(`M/D/YYYY`).

Before converting the column to a datetime data type, all records will be
tested to identify any values that cannot be interpreted as valid dates. This
prevents unexpected formatting problems from being hidden during conversion.

In [44]:
parsed_dates = pd.to_datetime(
    spotify_df["Release Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

invalid_dates = parsed_dates.isna().sum()

invalid_dates

np.int64(0)

### Converting Release Date to Datetime

All 4,593 release-date values were successfully validated using the
month/day/year format, with no invalid dates detected.

The `Release Date` column can therefore be safely converted from text to a
datetime data type. This will support later time-based analysis such as
examining release years, release months, track age, and changes in music
performance over time.

In [45]:
spotify_df["Release Date"] = pd.to_datetime(
    spotify_df["Release Date"],
    format="%m/%d/%Y"
)

In [46]:
spotify_df["Release Date"].dtype

dtype('<M8[us]')

### Release Date Conversion Result

The `Release Date` column was successfully converted from text to a datetime
data type.

All 4,593 records passed the date-format validation before conversion, so no
invalid dates were introduced. The dataset can now support time-based analysis
such as release-year trends, release-month patterns, and track-age calculations.

### 2.7 Cleaning the All Time Rank Column

The `All Time Rank` column represents the overall ranking position of each
track. Although ranking positions are numerical, the column is currently stored
as text.

The values will first be inspected to identify their formatting before the
column is converted to an appropriate numeric data type.


In [47]:
spotify_df["All Time Rank"].head(20)

0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      8
8      9
9     10
10    11
11    12
12    13
13    14
14    15
15    16
16    17
17    18
18    19
19    20
Name: All Time Rank, dtype: str

### Converting All Time Rank to Numeric

The `All Time Rank` column contains numerical ranking positions but is currently
stored as text.

The sample values show valid integer rankings, so the column will be converted
to a numeric data type. Any formatting characters such as commas will be
removed before conversion.

In [48]:
spotify_df["All Time Rank"] = pd.to_numeric(
    spotify_df["All Time Rank"].str.replace(",", "", regex=False),
    errors="coerce"
)

In [49]:
spotify_df["All Time Rank"].dtype

dtype('int64')

In [50]:
spotify_df["All Time Rank"].isnull().sum()

np.int64(0)

### All Time Rank Conversion Result

The `All Time Rank` column was successfully converted from text to an integer
data type.

All ranking values were converted successfully, with no missing values
introduced during the process. The column can now be used correctly for
numerical sorting, ranking comparisons, statistical analysis, and later
feature engineering.

### 2.8 Checking Text Columns for Whitespace Issues

The key text columns will now be checked for leading or trailing whitespace.

This is important because hidden spaces can prevent correct matching between
records. For example, `Taylor Swift` and `Taylor Swift ` may look similar but
would be treated as different values by the computer.

The columns checked are `Track`, `Album Name`, `Artist`, and `ISRC`.


In [51]:
text_columns = ["Track", "Album Name", "Artist", "ISRC"]

whitespace_summary = {}

for column in text_columns:
    whitespace_summary[column] = (
        spotify_df[column] != spotify_df[column].str.strip()
    ).sum()

whitespace_summary

{'Track': np.int64(25),
 'Album Name': np.int64(24),
 'Artist': np.int64(9),
 'ISRC': np.int64(0)}

### Inspecting Records with Extra Whitespace

The whitespace check identified leading or trailing spaces in the `Track`,
`Album Name`, and `Artist` columns, while no whitespace issues were found in
`ISRC`.

Before modifying the data, the affected records will be inspected to confirm
the formatting issue.

In [52]:
whitespace_mask = (
    (spotify_df["Track"] != spotify_df["Track"].str.strip()) |
    (spotify_df["Album Name"] != spotify_df["Album Name"].str.strip()) |
    (spotify_df["Artist"] != spotify_df["Artist"].str.strip())
)

spotify_df.loc[
    whitespace_mask,
    ["Track", "Album Name", "Artist", "ISRC"]
]

,Track,Album Name,Artist,ISRC
351,Lï¿½ï¿½ï¿½AMOUR DE,HIT ME HARD AND SOFT,Billie Eilish,USUM72401990
531,Tï¿½ï¿½,Dolido Pero No Arrepentido - EP,Fuerza Regida,QZ9QQ2400125
604,Cï¿½ï¿½ï¿½t ï¿½ï¿½ï¿½ï¿,Cï¿½ï¿½ï¿½t ï¿½ï¿½ï¿½ï¿,Tï¿½ï¿½ng Duy,QZYHM2362070
705,Por Mi Mexico (Remix),"Por Mi Mï¿½ï¿½xico (Remix) [feat. Dharius, C-K...",Lefty Sm,QM3DF2219446
846,Diz Aï¿½ï¿½ Qual ï¿½ï¿½,Diz Aï¿½ï¿½ Qual ï¿½ï¿½,Mc IG,BRWMB2400220
876,Tek Baï¿½ï¿½,Tek Baï¿½ï¿½ï¿½ï¿½ma,Semicenk,FRX452438249
1038,Bad Memories (feat. Elley Duhï¿½ï¿½ & FAST,Bad Memories (feat. Elley Duhï¿½ï¿½ & FAST,MEDUZA,GBUM72203666
1058,Canï¿½ï¿½ï¿½t Catch Me Now - from The Hunger G...,Canï¿½ï¿½ï¿½t Catch Me Now (from The Hunger Ga...,Olivia Rodrigo,USUG12307028
1078,Youï¿½ï¿½ï¿½re Mines Still (feat.,Youï¿½ï¿½ï¿½re Mines Still (feat.,Yung Bleu,USWB12004394
1284,Bï¿½ï¿½cane - A COLORS,Bï¿½ï¿½cane - A Colors Show - Si,Yamï¿,DEXC82300032


### Removing Leading and Trailing Whitespace

The inspection confirmed that some values in the `Track`, `Album Name`, and
`Artist` columns contain unnecessary leading or trailing whitespace.

These spaces do not provide meaningful information and may cause problems
when comparing, grouping, filtering, or joining records. The affected text
columns will therefore be standardised using `str.strip()`.

The `ISRC` column does not require modification because no whitespace issues
were detected.

In [53]:
columns_to_strip = ["Track", "Album Name", "Artist"]

for column in columns_to_strip:
    spotify_df[column] = spotify_df[column].str.strip()

In [54]:
whitespace_check = {}

for column in ["Track", "Album Name", "Artist", "ISRC"]:
    whitespace_check[column] = (
        spotify_df[column] != spotify_df[column].str.strip()
    ).sum()

whitespace_check

{'Track': np.int64(0),
 'Album Name': np.int64(0),
 'Artist': np.int64(0),
 'ISRC': np.int64(0)}

### 2.9 Investigating Text Encoding Issues

Some track, album, and artist names contain unusual replacement characters,
such as `¿½`, which suggests that certain text values were decoded incorrectly
when the source CSV was loaded.

Before modifying these values, the dataset will be checked to determine how
many records contain suspicious encoding patterns and which text columns are
affected.


In [55]:
encoding_issue_summary = {}

for column in ["Track", "Album Name", "Artist"]:
    encoding_issue_summary[column] = (
        spotify_df[column]
        .str.contains("¿½", na=False)
        .sum()
    )

encoding_issue_summary

{'Track': np.int64(270), 'Album Name': np.int64(305), 'Artist': np.int64(115)}

### Investigating the Source Encoding

The encoding check identified suspicious character sequences in 270 track
names, 305 album names, and 115 artist names.

Because these issues affect a substantial number of records, manually replacing
individual characters would be unreliable and could alter legitimate names.

The source file will therefore be investigated to determine whether the issue
originates from the character encoding used when loading the CSV.

In [56]:
from pathlib import Path

spotify_file = Path("../data/raw/Most Streamed Spotify Songs 2024.csv.zip")

raw_bytes = spotify_file.read_bytes()

raw_bytes[:100]

b'PK\x03\x04-\x00\x00\x08\x08\x00Z\x96\xcfX\xfb\xd4\xde\xc9\xff\xff\xff\xff\xff\xff\xff\xff$\x00\x14\x00Most Streamed Spotify Songs 2024.csv\x01\x00\x10\x00I\xc0\x10\x00\x00\x00\x00\x00\x14\xc0\x07\x00\x00\x00\x00\x00\xac\xbd\xebr\x1bY\x92\xa5\xfb\x7f\xcc\xfa\x1db'

### Reading the CSV Content Inside the ZIP Archive

The raw ZIP file begins with the expected `PK` archive signature, confirming
that the previous inspection was reading compressed binary data rather than the
CSV text itself.

The CSV file inside the archive will now be opened directly so that its raw
text bytes can be inspected before testing alternative character encodings.

In [57]:
import zipfile

with zipfile.ZipFile(
    "../data/raw/Most Streamed Spotify Songs 2024.csv.zip",
    "r"
) as zip_file:
    
    with zip_file.open("Most Streamed Spotify Songs 2024.csv") as csv_file:
        csv_bytes = csv_file.read(5000)

csv_bytes[:500]

b'Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,YouTube Playlist Reach,Apple Music Playlist Count,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track\r\nMILLION DOLLAR BABY,Million Dollar'

### Testing UTF-8 Decoding

The CSV content inside the ZIP archive can now be accessed directly. The
beginning of the file contains standard ASCII characters, which can be decoded
successfully by several different character encodings.

To determine whether the original dataset uses UTF-8, the complete CSV content
will be read and an attempt will be made to decode it using UTF-8.

In [58]:
with zipfile.ZipFile(
    "../data/raw/Most Streamed Spotify Songs 2024.csv.zip",
    "r"
) as zip_file:
    
    with zip_file.open("Most Streamed Spotify Songs 2024.csv") as csv_file:
        full_csv_bytes = csv_file.read()

try:
    utf8_text = full_csv_bytes.decode("utf-8")
    print("UTF-8 decoding successful.")
except UnicodeDecodeError as error:
    print("UTF-8 decoding failed.")
    print(error)

UTF-8 decoding failed.
'utf-8' codec can't decode byte 0xfd in position 2679: invalid start byte


### Testing Alternative Character Encodings

UTF-8 decoding failed because the source CSV contains byte sequences that are
not valid UTF-8.

This suggests that the dataset uses a different character encoding. Several
common single-byte encodings will therefore be tested before deciding how the
text should be loaded.

In [59]:
encodings_to_test = [
    "latin-1",
    "cp1252",
    "iso-8859-15"
]

for encoding in encodings_to_test:
    try:
        decoded_text = full_csv_bytes.decode(encoding)
        print(f"{encoding}: decoding successful")
    except UnicodeDecodeError as error:
        print(f"{encoding}: decoding failed")
        print(error)

latin-1: decoding successful
cp1252: decoding successful
iso-8859-15: decoding successful


### Comparing Alternative Encodings

Several single-byte encodings can successfully decode the source file.
However, successful decoding alone does not confirm that an encoding correctly
represents the original text.

A section of the CSV containing non-ASCII characters will therefore be decoded
using each candidate encoding. Comparing the resulting text will help determine
which encoding produces the most meaningful artist, track, and album names.

In [60]:
start = 2600
end = 2800

sample_bytes = full_csv_bytes[start:end]

for encoding in encodings_to_test:
    print(f"\n--- {encoding} ---")
    print(sample_bytes.decode(encoding))


--- latin-1 ---
0,87,"33,245,595",53,"3,372,428","5,762",,"6,063,523",,1
Danza Kuduro - Cover,ýýýýýýýýýýýýýýýýýýýýý - ýýýýýýýýýýýýýýýýýý -,MUSIC LAB JPN,6/9/2024,TCJPA2463708,9,355.7,"1,653,018,119",1,15,,,,,,,,,,,,

--- cp1252 ---
0,87,"33,245,595",53,"3,372,428","5,762",,"6,063,523",,1
Danza Kuduro - Cover,ýýýýýýýýýýýýýýýýýýýýý - ýýýýýýýýýýýýýýýýýý -,MUSIC LAB JPN,6/9/2024,TCJPA2463708,9,355.7,"1,653,018,119",1,15,,,,,,,,,,,,

--- iso-8859-15 ---
0,87,"33,245,595",53,"3,372,428","5,762",,"6,063,523",,1
Danza Kuduro - Cover,ýýýýýýýýýýýýýýýýýýýýý - ýýýýýýýýýýýýýýýýýý -,MUSIC LAB JPN,6/9/2024,TCJPA2463708,9,355.7,"1,653,018,119",1,15,,,,,,,,,,,,


### Encoding Investigation Result

Multiple character encodings were tested to determine whether the unusual
characters found in some track, album, and artist names were caused by using an
incorrect encoding when loading the dataset.

UTF-8 decoding failed, while `latin-1`, `cp1252`, and `iso-8859-15` were all
able to decode the source file. However, each encoding produced the same
corrupted character sequences within the affected records.

This indicates that the text corruption is likely already present in the
source dataset rather than being introduced during loading.

Because the original characters cannot be recovered reliably from the
available file, these values will be retained rather than manually guessed or
replaced. Records may later be enriched using reliable external identifiers
such as ISRC if accurate metadata recovery is required.

In [61]:
full_csv_bytes.count(b"\xfd")

4050

### Source Encoding Validation

The raw CSV contains 4,050 occurrences of the byte `0xFD`, which corresponds
to the unusual characters observed in many affected track, album, and artist
names when decoded using the available single-byte encodings.

This provides further evidence that the character corruption is embedded in
the source dataset itself rather than being introduced by Pandas or the
cleaning process.

The affected text values will therefore be retained in their source form.
Recovering the original metadata would require a separate data-enrichment
process using reliable external identifiers such as ISRC.

### 2.11 Final Validation and Save

The main cleaning operations have now been completed. Before saving the
processed Spotify dataset, a final validation will be performed to confirm
that the dataset remains structurally consistent and that the cleaning
operations produced the expected results.

The validation will examine the final dataset shape, duplicate records,
missing values, data types, key identifier fields, and cleaned text columns.


In [62]:
print("Final dataset shape:")
print(spotify_df.shape)

print("\nDuplicate rows:")
print(spotify_df.duplicated().sum())

print("\nMissing Artist values:")
print(spotify_df["Artist"].isnull().sum())

print("\nTIDAL Popularity present:")
print("TIDAL Popularity" in spotify_df.columns)

Final dataset shape:
(4593, 28)

Duplicate rows:
0

Missing Artist values:
0

TIDAL Popularity present:
False


### Final Data Type Validation

The data types are checked again after cleaning to confirm that each variable
is stored in a suitable format for subsequent analysis. In particular,
release dates should be represented as datetime values, ranking variables as
integers, and streaming and engagement metrics as numeric values.

In [63]:
spotify_df.dtypes

Track                                    str
Album Name                               str
Artist                                   str
Release Date                  datetime64[us]
ISRC                                     str
All Time Rank                          int64
Track Score                          float64
Spotify Streams                      float64
Spotify Playlist Count               float64
Spotify Playlist Reach               float64
Spotify Popularity                   float64
YouTube Views                        float64
YouTube Likes                        float64
TikTok Posts                         float64
TikTok Likes                         float64
TikTok Views                         float64
YouTube Playlist Reach               float64
Apple Music Playlist Count           float64
AirPlay Spins                        float64
SiriusXM Spins                       float64
Deezer Playlist Count                float64
Deezer Playlist Reach                float64
Amazon Pla

### Data Type Validation Interpretation

The final data type inspection confirms that the dataset is now stored in suitable formats for analysis. Text-based variables such as track, album, artist and ISRC are represented as strings, while Release Date has been converted to a datetime format.

All Time Rank and Explicit Track are stored as integers, while Track Score and the streaming, playlist, popularity and engagement metrics are stored as numeric values. Several count-based variables remain as float values because they contain missing observations, allowing Pandas to represent the missing entries as NaN.

Overall, the dataset now has appropriate data types for the next stages of analysis.

### 2.10 Numeric Range and Missingness Analysis

After converting the relevant variables into appropriate numeric data types, the
values are inspected to identify potentially invalid or unexpected observations.

The Explicit Track variable is checked first because it represents a binary
category, where 0 indicates a non-explicit track and 1 indicates an explicit
track.


In [64]:
spotify_df["Explicit Track"].value_counts(dropna=False).sort_index()

Explicit Track
0    2942
1    1651
Name: count, dtype: int64

In [65]:
print("Minimum Spotify Popularity:")
print(spotify_df["Spotify Popularity"].min())

print("\nMaximum Spotify Popularity:")
print(spotify_df["Spotify Popularity"].max())

print("\nValues below 0:")
print((spotify_df["Spotify Popularity"] < 0).sum())

print("\nValues above 100:")
print((spotify_df["Spotify Popularity"] > 100).sum())

print("\nMissing values:")
print(spotify_df["Spotify Popularity"].isnull().sum())

Minimum Spotify Popularity:
1.0

Maximum Spotify Popularity:
96.0

Values below 0:
0

Values above 100:
0

Missing values:
799


In [66]:
count_columns = [
    "Spotify Streams",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "Apple Music Playlist Count",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Pandora Streams",
    "Pandora Track Stations",
    "Soundcloud Streams",
    "Shazam Counts"
]

negative_counts = {}

for column in count_columns:
    negative_counts[column] = (spotify_df[column] < 0).sum()

negative_counts

{'Spotify Streams': np.int64(0),
 'Spotify Playlist Count': np.int64(0),
 'Spotify Playlist Reach': np.int64(0),
 'YouTube Views': np.int64(0),
 'YouTube Likes': np.int64(0),
 'TikTok Posts': np.int64(0),
 'TikTok Likes': np.int64(0),
 'TikTok Views': np.int64(0),
 'YouTube Playlist Reach': np.int64(0),
 'Apple Music Playlist Count': np.int64(0),
 'AirPlay Spins': np.int64(0),
 'SiriusXM Spins': np.int64(0),
 'Deezer Playlist Count': np.int64(0),
 'Deezer Playlist Reach': np.int64(0),
 'Amazon Playlist Count': np.int64(0),
 'Pandora Streams': np.int64(0),
 'Pandora Track Stations': np.int64(0),
 'Soundcloud Streams': np.int64(0),
 'Shazam Counts': np.int64(0)}

### Track Score Inspection

Track Score is inspected separately to understand its observed range and
distribution. Since the expected upper boundary is not assumed in advance,
summary statistics are examined before determining whether any values should
be considered unusual or invalid.

In [67]:
spotify_df["Track Score"].describe()

count    4593.000000
mean       41.840235
std        38.562767
min        19.400000
25%        23.300000
50%        29.900000
75%        44.400000
max       725.400000
Name: Track Score, dtype: float64

In [68]:
spotify_df[
    [
        "Track",
        "Artist",
        "All Time Rank",
        "Track Score",
        "Spotify Streams",
        "Spotify Popularity"
    ]
].sort_values(
    "Track Score",
    ascending=False
).head(20)

,Track,Artist,All Time Rank,Track Score,Spotify Streams,Spotify Popularity
0,MILLION DOLLAR BABY,Tommy Richman,1,725.4,3.904709e+08,92.0
1,Not Like Us,Kendrick Lamar,2,545.9,3.237039e+08,92.0
2,i like the way you kiss me,Artemas,3,538.4,6.013093e+08,92.0
3,Flowers,Miley Cyrus,4,444.9,2.031281e+09,85.0
4,Houdini,Eminem,5,423.3,1.070349e+08,88.0
5,Lovin On Me,Jack Harlow,6,410.1,6.706654e+08,83.0
6,Beautiful Things,Benson Boone,7,407.2,9.001588e+08,86.0
7,Gata Only,FloyyMenor,8,375.8,6.750792e+08,92.0
8,Danza Kuduro - Cover,MUSIC LAB JPN,9,355.7,1.653018e+09,NaN
9,BAND4BAND (feat. Lil Baby),Central Cee,10,330.6,9.067657e+07,86.0


In [69]:
print("Minimum rank:")
print(spotify_df["All Time Rank"].min())

print("\nMaximum rank:")
print(spotify_df["All Time Rank"].max())

print("\nRanks less than 1:")
print((spotify_df["All Time Rank"] < 1).sum())

print("\nUnique ranks:")
print(spotify_df["All Time Rank"].nunique())

print("\nDuplicate rank values:")
print(spotify_df["All Time Rank"].duplicated().sum())

Minimum rank:
1

Maximum rank:
4998

Ranks less than 1:
0

Unique ranks:
4573

Duplicate rank values:
20


In [70]:
duplicate_rank_rows = spotify_df[
    spotify_df["All Time Rank"].duplicated(keep=False)
][
    [
        "Track",
        "Artist",
        "All Time Rank",
        "Track Score",
        "Spotify Streams"
    ]
].sort_values("All Time Rank")

duplicate_rank_rows

,Track,Artist,All Time Rank,Track Score,Spotify Streams
310,Danza Kuduro - Cover,MUSIC LAB JPN,355,86.6,1.627430e+09
354,deja vu,Olivia Rodrigo,355,81.3,1.606976e+09
399,I DONï¿½ï¿½ï¿½T WANNA DO THIS A,Juliï¿½ï¿½n Kh,454,76.5,5.644448e+08
454,Unforgettable,French Montana,454,72.6,2.065697e+09
551,DIME QUE,Ian G.,559,65.4,NaN
562,3D (feat. Jack Harlow),Jung Kook,559,64.3,5.732490e+08
555,Broccoli,ati2x06,626,64.9,8.464359e+08
628,SICKO MODE,Travis Scott,626,60.9,2.134272e+09
1106,PRINCESITA DE ...,Jere Klein,1103,45.5,1.062902e+08
995,Cake By The Ocean - Cover,MUSIC LAB JPN,1103,48.2,1.611085e+09


### Missing Value Analysis

After validating the structure, data types, ranges, and duplicate records, the next step is to examine the remaining missing values in the dataset.

Missing values should not automatically be replaced with zero because a missing value does not necessarily mean that the actual value is zero. The percentage of missing data in each column will therefore be calculated first. This will help determine an appropriate treatment for each feature, such as retaining the missing values, imputing them, or removing a feature if it contains too much missing information.

In [71]:
missing_summary = pd.DataFrame({
    "Missing Values": spotify_df.isnull().sum(),
    "Missing Percentage": (
        spotify_df.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values(
    "Missing Percentage",
    ascending=False
)

missing_summary

,Missing Values,Missing Percentage
Soundcloud Streams,3327,72.44
SiriusXM Spins,2118,46.11
Pandora Track Stations,1263,27.50
TikTok Posts,1168,25.43
Pandora Streams,1101,23.97
Amazon Playlist Count,1050,22.86
YouTube Playlist Reach,1004,21.86
TikTok Views,976,21.25
TikTok Likes,975,21.23
Deezer Playlist Reach,923,20.10


### Investigating Patterns in Missing Data

The dataset contains different levels of missing information across streaming, playlist, radio, and social-media features. However, a missing value does not necessarily represent a value of zero.

Before applying any imputation or removing features, the missing data will be investigated to determine whether certain columns tend to be missing together. This is important because missing values may reflect differences in platform coverage rather than random data loss.

In [72]:
missing_columns = spotify_df.columns[
    spotify_df.isnull().any()
]

missing_correlation = (
    spotify_df[missing_columns]
    .isnull()
    .astype(int)
    .corr()
)

missing_correlation.round(2)

,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,YouTube Playlist Reach,Apple Music Playlist Count,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts
Spotify Streams,1.00,0.53,0.52,0.28,0.16,0.15,0.20,0.22,0.22,0.18,0.37,0.23,0.16,0.28,0.28,0.26,0.24,0.23,0.08,0.21
Spotify Playlist Count,0.53,1.00,0.98,0.26,0.24,0.23,0.19,0.22,0.22,0.17,0.29,0.25,0.13,0.23,0.23,0.20,0.19,0.17,0.07,0.18
Spotify Playlist Reach,0.52,0.98,1.00,0.26,0.25,0.24,0.20,0.22,0.22,0.18,0.29,0.26,0.13,0.23,0.23,0.20,0.20,0.18,0.07,0.18
Spotify Popularity,0.28,0.26,0.26,1.00,0.32,0.32,0.48,0.45,0.45,0.29,0.36,0.35,0.25,0.36,0.36,0.31,0.26,0.24,0.08,0.27
YouTube Views,0.16,0.24,0.25,0.32,1.00,0.99,0.29,0.32,0.32,0.50,0.34,0.44,0.25,0.30,0.30,0.26,0.24,0.22,0.06,0.37
YouTube Likes,0.15,0.23,0.24,0.32,0.99,1.00,0.29,0.32,0.32,0.50,0.34,0.43,0.25,0.30,0.30,0.27,0.25,0.23,0.07,0.36
TikTok Posts,0.20,0.19,0.20,0.48,0.29,0.29,1.00,0.86,0.86,0.31,0.29,0.29,0.37,0.33,0.32,0.26,0.33,0.33,0.08,0.27
TikTok Likes,0.22,0.22,0.22,0.45,0.32,0.32,0.86,1.00,1.00,0.33,0.32,0.31,0.37,0.35,0.35,0.29,0.35,0.35,0.09,0.30
TikTok Views,0.22,0.22,0.22,0.45,0.32,0.32,0.86,1.00,1.00,0.33,0.32,0.31,0.37,0.35,0.35,0.29,0.35,0.35,0.09,0.30
YouTube Playlist Reach,0.18,0.17,0.18,0.29,0.50,0.50,0.31,0.33,0.33,1.00,0.38,0.36,0.35,0.41,0.41,0.38,0.31,0.31,0.08,0.25


### Categorising Features by Missingness

The missing-data analysis shows strong relationships between several features from the same platforms. For example, YouTube Views and YouTube Likes have almost identical missingness patterns, while TikTok Views and TikTok Likes share the same pattern. Similar relationships are present between Spotify playlist features, Deezer playlist features, and Pandora features.

This suggests that some missing values may be related to platform-level data availability rather than random missing observations.

To guide the cleaning strategy, the numeric features will therefore be grouped according to their percentage of missing values. Features with low, moderate, high, and very high levels of missingness can then be treated separately rather than applying the same imputation method to every feature.

In [73]:
missing_percentage = spotify_df.isnull().mean() * 100

missing_groups = {
    "Low (< 10%)": missing_percentage[
        (missing_percentage > 0) &
        (missing_percentage < 10)
    ].index.tolist(),

    "Moderate (10-30%)": missing_percentage[
        (missing_percentage >= 10) &
        (missing_percentage < 30)
    ].index.tolist(),

    "High (30-50%)": missing_percentage[
        (missing_percentage >= 30) &
        (missing_percentage < 50)
    ].index.tolist(),

    "Very High (>= 50%)": missing_percentage[
        missing_percentage >= 50
    ].index.tolist()
}

for group, columns in missing_groups.items():
    print(f"\n{group}")
    print(columns)


Low (< 10%)
['Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'YouTube Views', 'YouTube Likes']

Moderate (10-30%)
['Spotify Popularity', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Shazam Counts']

High (30-50%)
['SiriusXM Spins']

Very High (>= 50%)
['Soundcloud Streams']


### Distribution and Skewness Analysis

Before selecting an imputation method, the distributions of the numeric features with missing values will be examined.

Streaming, view, like, playlist, and radio metrics may contain extremely large values for highly successful tracks, producing skewed distributions. In such cases, the mean may not provide an appropriate representation of a typical observation.

Skewness will therefore be calculated to help determine whether a more robust statistic, such as the median, would be more suitable for handling missing numeric values.

In [74]:
numeric_missing_columns = spotify_df.select_dtypes(
    include="number"
).columns[
    spotify_df.select_dtypes(include="number").isnull().any()
]

skewness_summary = (
    spotify_df[numeric_missing_columns]
    .skew()
    .sort_values(ascending=False)
)

skewness_summary

TikTok Likes                  32.051572
TikTok Views                  31.010298
Shazam Counts                 18.474349
TikTok Posts                   7.626266
Pandora Track Stations         6.716012
Deezer Playlist Reach          6.118403
SiriusXM Spins                 6.074749
YouTube Views                  6.024589
AirPlay Spins                  5.166020
Deezer Playlist Count          4.792900
Soundcloud Streams             4.370514
YouTube Likes                  4.222139
YouTube Playlist Reach         3.681932
Pandora Streams                3.151191
Apple Music Playlist Count     2.887222
Spotify Playlist Reach         2.595763
Amazon Playlist Count          2.186540
Spotify Streams                2.023892
Spotify Playlist Count         1.836000
Spotify Popularity            -2.051234
dtype: float64

### Handling Features with Very High Missingness

The missing-value and distribution analyses showed that the numeric platform metrics are generally highly skewed and that several features from the same platforms share similar missingness patterns.

Soundcloud Streams contains 72.44% missing values, meaning that most observations for this feature are unavailable. Imputing such a large proportion of the column could introduce substantial artificial information into the dataset. Therefore, Soundcloud Streams will be removed.

The remaining features will be retained at this stage. Missing values will not be globally replaced with zero because missing platform data does not necessarily represent zero activity. Imputation for machine-learning features can instead be performed later as part of the modelling pipeline, helping to avoid data leakage between training and test data.

In [75]:
spotify_df = spotify_df.drop(
    columns=["Soundcloud Streams"]
)

spotify_df.shape

(4593, 27)

### Final Data Quality Validation

After completing the cleaning process, a final validation is performed to confirm the structure and quality of the dataset.

This check verifies the final dataset dimensions, duplicate records, missing values, data types, and key numeric constraints. The purpose is to ensure that the transformations performed during cleaning were applied correctly before the cleaned dataset is saved for exploratory data analysis and later machine-learning tasks.

In [76]:
print("FINAL DATASET VALIDATION")
print("=" * 40)

# Dataset dimensions
print("\nDataset shape:")
print(spotify_df.shape)

# Duplicate rows
print("\nDuplicate rows:")
print(spotify_df.duplicated().sum())

# Missing values
print("\nTotal missing values:")
print(spotify_df.isnull().sum().sum())

print("\nColumns containing missing values:")
print(
    spotify_df.isnull()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

# Data types
print("\nData type summary:")
print(spotify_df.dtypes.value_counts())

# Explicit Track validation
print("\nExplicit Track values:")
print(spotify_df["Explicit Track"].value_counts().sort_index())

# Release Date validation
print("\nMissing Release Dates:")
print(spotify_df["Release Date"].isnull().sum())

# All Time Rank validation
print("\nInvalid All Time Rank values:")
print((spotify_df["All Time Rank"] < 1).sum())

# Spotify Popularity range
print("\nSpotify Popularity range:")
print(
    spotify_df["Spotify Popularity"].min(),
    "to",
    spotify_df["Spotify Popularity"].max()
)

FINAL DATASET VALIDATION

Dataset shape:
(4593, 27)

Duplicate rows:
0

Total missing values:
14771

Columns containing missing values:
SiriusXM Spins                2118
Pandora Track Stations        1263
TikTok Posts                  1168
Pandora Streams               1101
Amazon Playlist Count         1050
YouTube Playlist Reach        1004
TikTok Views                   976
TikTok Likes                   975
Deezer Playlist Reach          923
Deezer Playlist Count          916
Spotify Popularity             799
Shazam Counts                  576
Apple Music Playlist Count     556
AirPlay Spins                  493
YouTube Likes                  310
YouTube Views                  303
Spotify Streams                108
Spotify Playlist Reach          67
Spotify Playlist Count          65
dtype: int64

Data type summary:
float64           20
str                4
int64              2
datetime64[us]     1
Name: count, dtype: int64

Explicit Track values:
Explicit Track
0    2942
1    16

### Saving the Cleaned Dataset

The final cleaned dataset is exported as a new CSV file so that the original raw dataset remains unchanged.

This cleaned version will be used in the next stages of the project, including exploratory data analysis, data visualisation, feature engineering, and machine-learning model development.

In [77]:
from pathlib import Path

cleaned_data_path = Path("../data/processed")
cleaned_data_path.mkdir(parents=True, exist_ok=True)

output_file = cleaned_data_path / "spotify_2024_cleaned.csv"

spotify_df.to_csv(
    output_file,
    index=False
)

print("Cleaned dataset saved successfully.")
print(f"Location: {output_file}")
print(f"Shape: {spotify_df.shape}")

Cleaned dataset saved successfully.
Location: ../data/processed/spotify_2024_cleaned.csv
Shape: (4593, 27)


# 3. Charts Dataset Cleaning

The charts dataset is the third and final raw dataset in the project. This section inspects, cleans, transforms, and validates the dataset before saving a processed version for later analysis.

In [78]:
import zipfile
from pathlib import Path

charts_zip_path = Path("../data/raw/charts.csv.zip")

with zipfile.ZipFile(charts_zip_path, "r") as zip_file:
    print("Files inside ZIP:")
    
    for file_name in zip_file.namelist():
        print(file_name)

Files inside ZIP:
charts.csv


### 3.1 Loading and Initial Inspection

Before cleaning the charts dataset, the data is loaded and inspected to understand its structure, dimensions, columns, data types, and overall content.

No modifications are made at this stage.


In [79]:
import pandas as pd

charts_df = pd.read_csv(
    "../data/raw/charts.csv.zip",
    compression="zip"
)

print("Charts dataset loaded successfully.")
print(f"Rows: {charts_df.shape[0]:,}")
print(f"Columns: {charts_df.shape[1]}")

Charts dataset loaded successfully.
Rows: 5,428,021
Columns: 10


### 3.2 Dataset Structure and Data Types

The charts dataset contains over 5.4 million observations. Before cleaning, the column names and a small sample of records are inspected to understand what each row represents and which variables are available.


In [80]:
print("Column names:")

for column in charts_df.columns:
    print(f"- {column}")

Column names:
- date
- country
- position
- streams
- track_id
- artists
- artist_genres
- duration
- explicit
- name


In [81]:
charts_df.head(10)

,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name
0,2021/04/15,de,82,625718,20IvMlpi4U5RuDnAlXSRiV,['Haftbefehl'],['german hip hop'],198746,False,Crackküche
1,2019/01/31,jp,171,50896,0V1K6MU0utODk4yNqZKsFv,['Suchmos'],"['japanese r&b', 'j-rock', 'japanese soul', 'j...",408320,False,WATER
2,2018/11/15,tr,59,185439,4qzZm5EIdFurBpDieEmVc9,['Nilipek.'],"['turkish singer-songwriter', 'turkish rock']",257142,False,Gözleri Aşka Gülen
3,2018/11/22,tr,133,111159,4qzZm5EIdFurBpDieEmVc9,['Nilipek.'],"['turkish singer-songwriter', 'turkish rock']",257142,False,Gözleri Aşka Gülen
4,2018/11/29,tr,166,96204,4qzZm5EIdFurBpDieEmVc9,['Nilipek.'],"['turkish singer-songwriter', 'turkish rock']",257142,False,Gözleri Aşka Gülen
5,2018/12/06,tr,184,90088,4qzZm5EIdFurBpDieEmVc9,['Nilipek.'],"['turkish singer-songwriter', 'turkish rock']",257142,False,Gözleri Aşka Gülen
6,2023/02/02,co,98,233388,6vNhBzDn5yU2fu67YliSgw,"['Kalido', 'Totoy El Frio', 'HIT$ MUSIC']","['reggaeton colombiano', 'urbano latino']",161653,True,Opciones
7,2017/02/23,hu,184,5389,1vWrdDoVve3adC23brWRke,"['Totova & Freddie Shuman', 'Begi Lotfi']",[],185806,False,Hosszú Idők
8,2014/11/16,hk,148,4645,0N8PEQ33Ba841py3SEV0Wp,['Jay Chou'],"['taiwan pop', 'zhongguo feng', 'mandopop', 'c...",280720,False,愛你沒差
9,2014/11/23,hk,103,6119,0N8PEQ33Ba841py3SEV0Wp,['Jay Chou'],"['taiwan pop', 'zhongguo feng', 'mandopop', 'c...",280720,False,愛你沒差


## Data Type Inspection

The data types of the charts dataset are inspected before cleaning. This helps identify columns that may require conversion, such as dates, numeric measurements, boolean values, or text-based categorical variables.

In [82]:
charts_df.dtypes

date               str
country            str
position         int64
streams          int64
track_id           str
artists            str
artist_genres      str
duration         int64
explicit          bool
name               str
dtype: object

### 3.3 Missing Values Inspection

Missing values are inspected across all columns before any cleaning decisions are made.

Both the number and percentage of missing values are calculated because the dataset contains over 5.4 million observations. This provides a clearer indication of the scale of missing data within each variable.


In [83]:
missing_values = charts_df.isnull().sum()

missing_percentage = (
    charts_df.isnull().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary.sort_values(
    "Missing Values",
    ascending=False
)

,Missing Values,Missing Percentage
name,885,0.02
date,0,0.00
country,0,0.00
position,0,0.00
streams,0,0.00
track_id,0,0.00
artists,0,0.00
artist_genres,0,0.00
duration,0,0.00
explicit,0,0.00


### Missing Values Observation

The charts dataset is highly complete. Only the `name` column contains missing values, with 885 missing observations out of more than 5.4 million records, representing approximately 0.02% of the dataset.

No rows are removed at this stage. Since the affected records still contain other potentially useful information, including `track_id`, artist, streams, chart position, and date, the missing track names will be investigated before deciding whether removal or another treatment is necessary.

In [84]:
missing_name_rows = charts_df[
    charts_df["name"].isna()
]

print(f"Rows with missing track names: {len(missing_name_rows):,}")

missing_name_rows.head(20)

Rows with missing track names: 885


,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name
142335,2020/03/26,py,194,16356,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142336,2020/04/30,py,169,20551,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142337,2020/05/07,py,173,19449,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142338,2020/05/14,py,179,19584,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142339,2020/05/21,py,191,19806,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142340,2020/06/11,py,194,20420,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142341,2020/06/18,py,188,21383,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142342,2020/06/25,py,180,22722,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142343,2020/07/02,py,191,18844,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN
142344,2020/07/16,py,150,24055,7xABntVHfPeFk622h4VFqk,['Various Artists'],[],0,False,NaN


In [85]:
# Count unique track IDs among rows with missing names
missing_track_ids = missing_name_rows["track_id"].unique()

print("Missing-name rows:")
print(len(missing_name_rows))

print("\nUnique track IDs with missing names:")
print(len(missing_track_ids))

print("\nTrack IDs:")
print(missing_track_ids)

Missing-name rows:
885

Unique track IDs with missing names:
94

Track IDs:
<StringArray>
['7xABntVHfPeFk622h4VFqk', '62bHyrzzw48JR3ex1hFqGW', '60pysSgEslc7i5blU5zZbS',
 '461JKAn7H6Sbx0ql9IvRUG', '5QarVtsSmGDiIsCy49yLEs', '3qUCIw0vwknzxksvu5OrFf',
 '5cjecvX0CmC9gK0Laf5EMQ', '2IfYatzvdtsB2wYvbT24AM', '4HhgmNcGxOqLWGE9HS2iXZ',
 '6ZpXDhe3keeGru2PBWfoMc', '0yCxbk63mOCcatXwrVTQ0d', '3jrEwnoMIdVjqMQcTB3B08',
 '096hrPBPEtvHbJg1N8mOfP', '1YqcGlCHNquxBhlUZsjhMT', '3GgMVwiFaQonxnA7PoR5hM',
 '6jRuGoydwAnzUvpSP3zXId', '5R6VPTr58f80Y5elByhQVP', '2TBoaEjn4wy7RAQSBzNSR9',
 '0zuqABAUKs8WLVq8vCmfst', '2IjVFm1CnORhXtsm1QkV0i', '1lROXJjQxzq35Zswlk96sO',
 '4u6Ib5oyD7dlz3xo0Ht7Gh', '3woM5duQfkDdoXwV5nJwXZ', '7A9apF3lxPLOO0qLPAF9Nx',
 '2Z2CqQXHgU00DxvnNLGeSr', '6Pdz4FCPs4O1ZRcejbfBrt', '1DmCs9mXUP9wnaTQcsT5aH',
 '5p7Zd8RAb4dLV1MK2owriM', '6jnVe0mrlljgBx7il7YtUJ', '3tHxZOR9pDLnsKGikqHqd8',
 '5KgRBx9jvzuQJlxc44WWVd', '0vatkYYyTAq7PSTP1EkIEV', '7vTU3jaddNDNycENrbH690',
 '3ZDj9ogLWLHgUkSfmyzSbl', '47rbjDud83d3a

In [86]:
# Search the full dataset for valid names belonging to
# track IDs that appear in the missing-name rows

recoverable_names = charts_df[
    charts_df["track_id"].isin(missing_track_ids)
    & charts_df["name"].notna()
][
    ["track_id", "name", "artists"]
].drop_duplicates()

print("Matching records with valid track names:")
print(len(recoverable_names))

recoverable_names

Matching records with valid track names:
0


,track_id,name,artists


### Handling Missing Track Names

The `name` column contained 885 missing values. Investigation showed that these rows represented 94 unique track IDs, meaning that some unidentified tracks appeared repeatedly across different chart dates and countries.

The full dataset was searched to determine whether valid track names existed elsewhere for the same track IDs. No matching records with valid names were found, so the missing track names could not be reliably recovered from the available data.

Since the 885 affected rows represent only a very small proportion of the dataset (approximately 0.02%), they were removed rather than assigning artificial or potentially incorrect track names. This preserves the reliability of track-level analysis while having a negligible effect on the overall dataset size.

In [87]:
# Record dataset size before removing missing track names
rows_before = len(charts_df)

# Remove rows where the track name is missing
charts_df = charts_df.dropna(subset=["name"]).copy()

# Record dataset size after cleaning
rows_after = len(charts_df)

print("Rows before:", f"{rows_before:,}")
print("Rows removed:", f"{rows_before - rows_after:,}")
print("Rows after:", f"{rows_after:,}")

print("\nRemaining missing track names:")
print(charts_df["name"].isna().sum())

Rows before: 5,428,021
Rows removed: 885
Rows after: 5,427,136

Remaining missing track names:
0


### 3.4 Duplicate Row Inspection

The charts dataset contains repeated observations of the same tracks because a track can appear on charts across multiple dates, countries, and chart positions. Therefore, repeated track IDs are expected and should not automatically be treated as duplicates.

This step checks for exact duplicate rows, where all column values are identical. Exact duplicates may represent redundant records and can be removed without losing unique chart observations.


In [88]:
# Check for exact duplicate rows
duplicate_count = charts_df.duplicated().sum()

print("Total rows:", f"{len(charts_df):,}")
print("Exact duplicate rows:", f"{duplicate_count:,}")

if duplicate_count > 0:
    print("\nPercentage of rows that are exact duplicates:")
    print(f"{(duplicate_count / len(charts_df)) * 100:.4f}%")

Total rows: 5,427,136
Exact duplicate rows: 0


### 3.5 Date Validation and Conversion

The `date` column is currently stored as a string rather than a datetime datatype. Since the dataset contains chart observations over time, converting this column to datetime will make chronological analysis, filtering, grouping, and visualisation more reliable.

Before permanently converting the column, the date values are first parsed using Pandas. Any value that cannot be interpreted as a valid date will be converted temporarily to a missing datetime value (`NaT`). This allows invalid date entries to be identified before modifying the original column.


In [89]:
# Temporarily parse the date column to identify invalid dates
parsed_dates = pd.to_datetime(
    charts_df["date"],
    errors="coerce"
)

invalid_dates = parsed_dates.isna().sum()

print("Total rows:", f"{len(charts_df):,}")
print("Invalid or unparseable dates:", f"{invalid_dates:,}")

if invalid_dates > 0:
    print("\nExamples of invalid date values:")
    print(
        charts_df.loc[
            parsed_dates.isna(),
            "date"
        ].value_counts().head(20)
    )

Total rows: 5,427,136
Invalid or unparseable dates: 1,968

Examples of invalid date values:
date
2013-05-26    161
2013-06-02    158
2013-04-28    153
2013-05-05    144
2013-05-19    135
2013-06-09    135
2013-05-12    132
2013-06-23    109
2013-06-16    106
2013-06-30     77
2013-07-07     62
2013-07-14     50
2013-07-21     38
2013-07-28     23
2013-08-04     12
2019-12-19     10
2019-12-26     10
2019-04-18      8
2019-11-21      8
2019-11-28      8
Name: count, dtype: int64


### Investigation of Mixed Date Formats

The initial date conversion identified 1,968 values as unparseable. However, inspection showed that these values appear to be valid dates, such as `2013-05-26` and `2019-12-19`.

The dataset contains dates represented using different separators, including formats such as `YYYY/MM/DD` and `YYYY-MM-DD`. Therefore, the flagged values may result from mixed date formatting rather than genuinely invalid dates.

Before removing any records, the date column will be parsed again using Pandas' mixed-format date handling. This allows each value to be interpreted according to its individual date format.

In [90]:
# Parse dates while allowing mixed date formats
mixed_parsed_dates = pd.to_datetime(
    charts_df["date"],
    format="mixed",
    errors="coerce"
)

mixed_invalid_dates = mixed_parsed_dates.isna().sum()

print("Total rows:", f"{len(charts_df):,}")
print("Invalid dates after mixed-format parsing:", f"{mixed_invalid_dates:,}")

if mixed_invalid_dates > 0:
    print("\nRemaining invalid date values:")
    print(
        charts_df.loc[
            mixed_parsed_dates.isna(),
            "date"
        ].value_counts().head(20)
    )

Total rows: 5,427,136
Invalid dates after mixed-format parsing: 0


### Date Column Conversion

The mixed-format investigation confirmed that all date values are valid when the different date formats are handled correctly. The earlier 1,968 unparseable values were therefore caused by inconsistent date formatting rather than invalid data.

Since no invalid dates remain, the `date` column can now be safely converted from a string to Pandas' datetime datatype using mixed-format parsing. This standardises the representation of dates and prepares the dataset for time-based analysis, filtering, grouping, and visualisation.

In [91]:
# Convert the date column permanently to datetime
charts_df["date"] = pd.to_datetime(
    charts_df["date"],
    format="mixed",
    errors="raise"
)

print("Date conversion completed successfully.")
print("\nDate datatype:")
print(charts_df["date"].dtype)

print("\nEarliest chart date:")
print(charts_df["date"].min())

print("\nLatest chart date:")
print(charts_df["date"].max())

print("\nMissing dates:")
print(charts_df["date"].isna().sum())

Date conversion completed successfully.

Date datatype:
datetime64[us]

Earliest chart date:
2013-04-28 00:00:00

Latest chart date:
2023-04-06 00:00:00

Missing dates:
0


### 3.6 Country, Chart Position, and Stream Validation

The `country`, `position`, and `streams` columns contain important information about each chart observation. Before using these variables for analysis, their values must be checked for possible inconsistencies or invalid ranges.

The `country` column will be inspected to determine the number and format of unique country codes. The `position` column will be checked for its minimum and maximum ranking values, as chart positions should be positive. The `streams` column will also be checked for negative or zero values, since these may indicate unusual or invalid observations.

This inspection is performed before making any changes to the dataset.


In [92]:
# Inspect country values
print("Unique countries:")
print(charts_df["country"].nunique())

print("\nSample country codes:")
print(sorted(charts_df["country"].unique())[:30])


# Inspect chart positions
print("\nMinimum chart position:")
print(charts_df["position"].min())

print("\nMaximum chart position:")
print(charts_df["position"].max())

print("\nPositions less than 1:")
print((charts_df["position"] < 1).sum())


# Inspect stream values
print("\nMinimum streams:")
print(charts_df["streams"].min())

print("\nMaximum streams:")
print(charts_df["streams"].max())

print("\nNegative stream values:")
print((charts_df["streams"] < 0).sum())

print("\nZero stream values:")
print((charts_df["streams"] == 0).sum())

Unique countries:
77

Sample country codes:
['ad', 'ae', 'ar', 'at', 'au', 'be', 'bg', 'bo', 'br', 'by', 'ca', 'ch', 'cl', 'co', 'cr', 'cy', 'cz', 'de', 'dk', 'do', 'ec', 'ee', 'eg', 'es', 'fi', 'fr', 'gb', 'global', 'gr', 'gt']

Minimum chart position:
1

Maximum chart position:
358

Positions less than 1:
0

Minimum streams:
0

Maximum streams:
115156896

Negative stream values:
0

Zero stream values:
7


###  Investigation of Unusual Chart Positions and Zero Stream Values

Initial validation showed that chart positions range from 1 to 358, with no values below 1. Stream counts are also non-negative, although seven observations contain zero streams.

Since values above position 200 and zero stream counts may still represent legitimate observations in the source dataset, these records should not be removed automatically.

This step investigates the unusual observations in more detail by examining their frequency, countries, dates, tracks, and associated stream values before deciding whether any cleaning action is necessary.

In [93]:
# Investigate chart positions above 200
positions_above_200 = charts_df[charts_df["position"] > 200]

print("Rows with chart position above 200:")
print(f"{len(positions_above_200):,}")

print("\nCountries containing positions above 200:")
print(
    positions_above_200["country"]
    .value_counts()
    .head(20)
)

print("\nPosition range above 200:")
print(
    positions_above_200["position"]
    .agg(["min", "max"])
)


# Investigate zero-stream observations
zero_stream_rows = charts_df[charts_df["streams"] == 0]

print("\n" + "=" * 50)

print("\nRows with zero streams:")
print(len(zero_stream_rows))

print("\nZero-stream observations:")
display(
    zero_stream_rows[
        [
            "date",
            "country",
            "position",
            "streams",
            "track_id",
            "artists",
            "name"
        ]
    ]
)

Rows with chart position above 200:
24,914

Countries containing positions above 200:
country
tw    1468
pl    1309
br    1200
ar    1155
ca    1080
ch    1058
mx    1040
es     939
be     919
us     854
sg     854
tr     842
fr     813
ph     793
co     783
pt     776
gb     741
cl     696
fi     684
se     680
Name: count, dtype: int64

Position range above 200:
min    201
max    358
Name: position, dtype: int64


Rows with zero streams:
7

Zero-stream observations:


,date,country,position,streams,track_id,artists,name
148628,2021-07-29,in,106,0,2rRJrJEo19S2J82BDsQ3F7,['Trevor Daniel'],Falling
1022573,2021-07-15,se,150,0,1qUYbmIfoIDVRaRuDUlVZ3,['Tjuvjakt'],Vaskar mina tårar
2362707,2021-06-10,kr,101,0,31qCy5ZaophVA81wtlwLc4,['Justin Bieber'],Anyone
2666623,2021-08-12,ph,131,0,00mBzIWv5gHOYxwuEJXjOG,['December Avenue'],Sa Ngalan Ng Pag-Ibig
2670971,2021-07-01,cy,55,0,3H7ihDc1dqLriiWXwsc2po,"['Topic', 'A7S']",Breaking Me
2911100,2021-09-02,hn,168,0,5lAnYvAIkSDNXqfo7DyFUm,['Doja Cat'],Ain't Shit
3413680,2021-06-17,be,17,0,5nujrmhLynf4yMoMtj8AQF,"['Dua Lipa', 'DaBaby']",Levitating (feat. DaBaby)


###  Validation Decision for Chart Positions and Stream Counts

Further investigation was carried out on unusual chart positions and zero stream values.

A total of 24,914 observations had chart positions above 200. These observations were distributed across multiple countries rather than being isolated to a single market. The maximum observed chart position was 358, and no positions below 1 were found. Therefore, positions above 200 were retained because there was insufficient evidence to classify them as invalid.

Seven observations contained a stream count of zero. Inspection showed that these records still contained valid-looking dates, countries, chart positions, track IDs, artists, and track names. Since zero streams may represent legitimate observations and there was no evidence that these rows were corrupted, they were also retained.

No rows were removed during this validation step.

### 3.7 Track Identifier and Text Field Validation

The dataset contains several important textual and identifier fields, including track IDs, artist information, genres, and track names.

Although the earlier missing-value analysis did not identify missing values in most of these columns, empty strings or whitespace-only values may not be detected as standard missing values.

This step therefore checks the main textual fields for blank or whitespace-only values and examines the uniqueness of track identifiers.


In [94]:
# Important text-based columns
text_columns = [
    "track_id",
    "artists",
    "artist_genres",
    "name"
]

print("Blank or whitespace-only values:")
print("-" * 40)

for column in text_columns:
    blank_count = (
        charts_df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{column}: {blank_count:,}")


print("\nTrack ID validation:")
print("-" * 40)

print(f"Total rows: {len(charts_df):,}")
print(f"Unique track IDs: {charts_df['track_id'].nunique():,}")
print(f"Missing track IDs: {charts_df['track_id'].isna().sum():,}")

Blank or whitespace-only values:
----------------------------------------
track_id: 0
artists: 0
artist_genres: 0
name: 0

Track ID validation:
----------------------------------------
Total rows: 5,427,136
Unique track IDs: 110,198
Missing track IDs: 0


### 3.7 Track Identifier and Text Field Validation Results

The validation found no blank or whitespace-only values in the `track_id`, `artists`, `artist_genres`, or `name` columns.

All 5,427,136 observations contain a track identifier, with 110,198 unique track IDs represented in the dataset. Repeated track IDs are expected because individual tracks can appear across multiple chart dates, countries, and positions.

No additional cleaning was required for these fields.


### 3.8 Duration and Explicit Content Validation

The `duration` and `explicit` columns are inspected to confirm that their values are reasonable and internally consistent.

For track duration, the analysis checks the minimum and maximum values as well as the number of zero and negative durations. The `explicit` column is also inspected to verify that it contains the expected Boolean categories.


In [95]:
print("Duration validation:")
print("-" * 40)

print(f"Minimum duration: {charts_df['duration'].min():,} ms")
print(f"Maximum duration: {charts_df['duration'].max():,} ms")

print(
    f"Zero durations: "
    f"{(charts_df['duration'] == 0).sum():,}"
)

print(
    f"Negative durations: "
    f"{(charts_df['duration'] < 0).sum():,}"
)

print("\nDuration summary:")
print(charts_df["duration"].describe())

print("\nExplicit content values:")
print("-" * 40)

print(charts_df["explicit"].value_counts(dropna=False))

Duration validation:
----------------------------------------
Minimum duration: 30,000 ms
Maximum duration: 9,318,296 ms
Zero durations: 0
Negative durations: 0

Duration summary:
count    5.427136e+06
mean     2.126294e+05
std      4.623193e+04
min      3.000000e+04
25%      1.850130e+05
50%      2.085120e+05
75%      2.337330e+05
max      9.318296e+06
Name: duration, dtype: float64

Explicit content values:
----------------------------------------
explicit
False    4003206
True     1423930
Name: count, dtype: int64


###  Investigation of Unusually Long Track Durations

The duration validation found no zero or negative values, and the typical track durations appear reasonable. However, the maximum recorded duration is substantially higher than the majority of observations.

Rather than treating long durations as errors automatically, the longest-duration records are inspected to determine whether they represent legitimate tracks or potentially anomalous data.

In [96]:
# Display the longest-duration tracks in the dataset

longest_tracks = (
    charts_df[
        ["track_id", "name", "artists", "duration"]
    ]
    .drop_duplicates(subset="track_id")
    .sort_values("duration", ascending=False)
    .head(20)
    .copy()
)

# Convert milliseconds to minutes for easier interpretation
longest_tracks["duration_minutes"] = (
    longest_tracks["duration"] / 1000 / 60
).round(2)

longest_tracks

,track_id,name,artists,duration,duration_minutes
2323897,6F8T0ZBpaYt9tTWlVGQRsr,Año Nuevo Colan 2017,['DJ Krlos Berrospi'],9318296,155.30
593895,2DEYFawpGha5Zn54Fx6dX5,Año Nuevo 2018,['DJ Krlos Berrospi'],7963609,132.73
2322037,0r0Qa0P3vD5wf7dDACz3qV,Festa (Mix Super Bailable),['DJ Luigi'],7215047,120.25
5368098,0cXWYaDMFTQjwcYB7PjGzg,Mix Primaveral 2016,['DJ Luigi'],3653956,60.90
4052541,6yob9nJN0rl9ASdZJxKk8H,Foxy Music,['Beginner'],2603773,43.40
3497731,4OdUZlOm3b3ociCXXz3btg,Reggaeton Karmoso 8,['DJ Luigi'],2429545,40.49
2063646,74vn4DEf2n5UgM0NRkxohb,Reggaeton Karmoso 9,['DJ Luigi'],2422518,40.38
1318829,0L6YK7xMLaxvf3tb5OVPXK,Unplugged Acustico,['Sech'],2147004,35.78
538904,1atrCmFrGEN5QvjoEWFRHo,SAS PLUS / SAS PUSSY,['Karpe'],1787030,29.78
1533672,7F2Eyu2eeannO90p7asSeK,Peru Al Mundial,['DJ Luigi'],1415063,23.58


###  Long-Duration Investigation Result

The longest track durations were manually inspected to determine whether the extreme values represented obvious data errors.

The inspection showed that the longest observations mainly correspond to extended mixes and long-form recordings. For example, the longest track has a duration of approximately 155 minutes, while other unusually long observations include recordings lasting approximately 133 and 120 minutes.

These values are therefore retained because there is insufficient evidence to classify them as erroneous. No zero or negative duration values were identified, so no duration records were removed or modified.

### 3.9 Categorical and Country Code Validation

The categorical fields are inspected to identify unusual or inconsistent values before finalising the cleaned dataset.

Country codes are examined to confirm the categories represented in the chart data. Artist and genre fields are also inspected to understand their structure and identify potentially unusual empty-list values.


In [97]:
# Inspect all country categories
print("Country values:")
print("-" * 50)

country_values = sorted(charts_df["country"].unique())

print(country_values)
print(f"\nTotal country categories: {len(country_values)}")


# Inspect artist and genre structure
print("\nArtist field examples:")
print("-" * 50)
print(charts_df["artists"].drop_duplicates().head(20).to_string(index=False))


print("\nArtist genre field examples:")
print("-" * 50)
print(charts_df["artist_genres"].drop_duplicates().head(20).to_string(index=False))


# Check for empty-list representations
print("\nEmpty-list values:")
print("-" * 50)

print(
    "artists == []:",
    (charts_df["artists"].str.strip() == "[]").sum()
)

print(
    "artist_genres == []:",
    (charts_df["artist_genres"].str.strip() == "[]").sum()
)

Country values:
--------------------------------------------------
['ad', 'ae', 'ar', 'at', 'au', 'be', 'bg', 'bo', 'br', 'by', 'ca', 'ch', 'cl', 'co', 'cr', 'cy', 'cz', 'de', 'dk', 'do', 'ec', 'ee', 'eg', 'es', 'fi', 'fr', 'gb', 'global', 'gr', 'gt', 'hk', 'hn', 'hu', 'id', 'ie', 'il', 'in', 'is', 'it', 'jp', 'kr', 'kz', 'lt', 'lu', 'lv', 'ma', 'mt', 'mx', 'my', 'ng', 'ni', 'nl', 'no', 'nz', 'pa', 'pe', 'ph', 'pk', 'pl', 'pt', 'py', 'ro', 'ru', 'sa', 'se', 'sg', 'sk', 'sv', 'th', 'tr', 'tw', 'ua', 'us', 'uy', 've', 'vn', 'za']

Total country categories: 77

Artist field examples:
--------------------------------------------------
                           ['Haftbefehl']
                              ['Suchmos']
                             ['Nilipek.']
['Kalido', 'Totoy El Frio', 'HIT$ MUSIC']
['Totova & Freddie Shuman', 'Begi Lotfi']
                             ['Jay Chou']
                                ['SANNI']
                           ['Alligatoah']
                         

### 3.9 Categorical and Country Code Validation Results

The categorical field inspection identified 77 country categories within the dataset, including individual country codes and a `global` chart category.

No empty artist lists were found, indicating that artist information is available for all remaining observations.

The `artist_genres` field contains 54,324 observations represented by an empty list (`[]`). These values indicate that genre metadata is unavailable for some tracks rather than representing conventional missing values. Since the affected observations still contain valid track, artist, chart, and streaming information, they are retained in the dataset rather than removed or assigned an artificial genre.

No categorical records were modified during this stage.


###  Country Code Structure Validation

The `country` field primarily uses two-letter country codes, with `global` representing worldwide chart observations.

The values are checked for structural consistency to identify unexpected categories, such as blank values, unusually long codes, or values that do not follow the expected two-letter format.

In [98]:
# Check whether country values follow the expected structure:
# two lowercase letters OR the special "global" category

country_pattern = r"^[a-z]{2}$"

invalid_country_values = charts_df.loc[
    ~charts_df["country"].str.match(country_pattern)
    & (charts_df["country"] != "global"),
    "country"
].value_counts()

print("Country structure validation:")
print("-" * 50)

print(f"Total country categories: {charts_df['country'].nunique()}")

print(
    "Two-letter country categories:",
    charts_df.loc[
        charts_df["country"].str.match(country_pattern),
        "country"
    ].nunique()
)

print(
    "Global observations:",
    (charts_df["country"] == "global").sum()
)

print("\nUnexpected country values:")
print(invalid_country_values)

print(
    "\nNumber of unexpected country categories:",
    len(invalid_country_values)
)

Country structure validation:
--------------------------------------------------
Total country categories: 77
Two-letter country categories: 76
Global observations: 88456

Unexpected country values:
Series([], Name: count, dtype: int64)

Number of unexpected country categories: 0


###  Country Code Structure Validation Results

The country field contains 77 categories. Of these, 76 follow the expected two-letter country-code structure, while `global` is used as a separate category for worldwide chart observations.

A total of 88,456 observations belong to the global chart category. No unexpected or structurally inconsistent country values were identified.

Therefore, no modifications were required for the `country` field.

### 3.10 Final Charts Dataset Validation

A final validation is performed after completing the cleaning process to confirm the overall quality and structure of the Charts dataset before it is saved.

The validation checks the final dataset dimensions, duplicate rows, missing values, date range, numeric ranges, categorical structure, and important identifier fields.


In [99]:
print("FINAL CHARTS DATASET VALIDATION")
print("=" * 55)

# Dataset shape
print("\nDataset shape:")
print(charts_df.shape)

# Duplicate rows
print("\nDuplicate rows:")
print(charts_df.duplicated().sum())

# Missing values
print("\nTotal missing values:")
print(charts_df.isna().sum().sum())

print("\nColumns containing missing values:")
missing_final = charts_df.isna().sum()
print(missing_final[missing_final > 0])

# Date validation
print("\nDate range:")
print("Earliest:", charts_df["date"].min())
print("Latest:", charts_df["date"].max())
print("Missing dates:", charts_df["date"].isna().sum())

# Track-name validation
print("\nMissing track names:")
print(charts_df["name"].isna().sum())

# Track ID validation
print("\nMissing track IDs:")
print(charts_df["track_id"].isna().sum())

print("Unique track IDs:")
print(charts_df["track_id"].nunique())

# Position validation
print("\nChart position range:")
print(
    charts_df["position"].min(),
    "to",
    charts_df["position"].max()
)

# Streams validation
print("\nStreams range:")
print(
    charts_df["streams"].min(),
    "to",
    charts_df["streams"].max()
)

print(
    "Negative streams:",
    (charts_df["streams"] < 0).sum()
)

# Duration validation
print("\nDuration range:")
print(
    charts_df["duration"].min(),
    "to",
    charts_df["duration"].max(),
    "ms"
)

print(
    "Negative durations:",
    (charts_df["duration"] < 0).sum()
)

# Country validation
print("\nCountry categories:")
print(charts_df["country"].nunique())

# Explicit validation
print("\nExplicit values:")
print(charts_df["explicit"].value_counts(dropna=False))

# Empty genre lists
print("\nEmpty artist genre lists:")
print(
    (charts_df["artist_genres"].str.strip() == "[]").sum()
)

FINAL CHARTS DATASET VALIDATION

Dataset shape:
(5427136, 10)

Duplicate rows:
0

Total missing values:
0

Columns containing missing values:
Series([], dtype: int64)

Date range:
Earliest: 2013-04-28 00:00:00
Latest: 2023-04-06 00:00:00
Missing dates: 0

Missing track names:
0

Missing track IDs:
0
Unique track IDs:
110198

Chart position range:
1 to 358

Streams range:
0 to 115156896
Negative streams: 0

Duration range:
30000 to 9318296 ms
Negative durations: 0

Country categories:
77

Explicit values:
explicit
False    4003206
True     1423930
Name: count, dtype: int64

Empty artist genre lists:
54324


### Saving the Cleaned Charts Dataset

Following the completion of the cleaning and validation process, the final Charts dataset is saved to the `data/processed` directory.

The cleaned dataset contains valid chart observations with standardised dates, complete track identifiers and names, validated numeric fields, and no conventional missing values or exact duplicate rows.

Empty genre lists are retained because they represent unavailable genre metadata rather than invalid observations.

In [100]:
from pathlib import Path

cleaned_data_path = Path("../data/processed")
cleaned_data_path.mkdir(parents=True, exist_ok=True)

output_file = cleaned_data_path / "charts_cleaned.csv"

charts_df.to_csv(
    output_file,
    index=False
)

print("Cleaned Charts dataset saved successfully.")
print(f"Location: {output_file}")
print(f"Shape: {charts_df.shape}")

Cleaned Charts dataset saved successfully.
Location: ../data/processed/charts_cleaned.csv
Shape: (5427136, 10)


# 4. Data Cleaning Summary

The three PMIP datasets have now been systematically inspected, cleaned and
validated in preparation for data integration.

The listeners dataset was checked for missing values, duplicates, numeric data
types and artist-name consistency. The cross-platform Spotify dataset required
additional treatment for duplicate records, missing artist information,
platform metrics, numeric conversion, dates, rankings, whitespace and source
encoding issues. The historical charts dataset was validated for missing track
names, duplicate observations, mixed date formats, chart positions, streams,
track identifiers, duration, explicit-content values and country categories.

The cleaned datasets are saved in `data/processed/` and are ready for the data
integration stage.
